# 01 — Problem and data contract

**Objectives**

- Define prediction time, unit, target horizon, positive label, and action.
- Make false-positive and false-negative assumptions visible.
- Review allowed and forbidden features before looking at model scores.

**Prerequisite:** lesson 00.


In [ ]:
import pandas as pd


from aai_local_classification.learning import study_root
from aai_local_classification.settings import load_settings

settings = load_settings()
root = study_root()
print(f"Course state: {root}")
print(f"Experiment: {settings.experiment_name}")


## Prediction contract

At a monthly account snapshot, predict whether a synthetic subscription will
churn within 30 days. `churned_30d = 1` is the positive event. A positive
prediction would send the account to a fictional retention review—not directly
take an action. Missing a churn is assigned five times the illustrative cost of
an unnecessary review.

These are authored learning assumptions. A real team must derive them with the
people who own the action, capacity, risk, and customer impact.


In [ ]:
contract = pd.Series(
    {
        "prediction_unit": "one account at one monthly snapshot",
        "prediction_time": "snapshot_date",
        "target": settings.data.target_column,
        "horizon": "30 days after the snapshot",
        "positive_label": 1,
        "primary_selection_metric": settings.selection.primary_metric,
        "false_negative_cost": settings.selection.false_negative_cost,
        "false_positive_cost": settings.selection.false_positive_cost,
    }
)
contract.to_frame("declared value")


In [ ]:
pd.DataFrame(
    {
        "role": (
            ["numeric feature"] * len(settings.features.numeric)
            + ["categorical feature"] * len(settings.features.categorical)
            + ["forbidden"] * len(settings.features.forbidden)
        ),
        "column": (
            list(settings.features.numeric)
            + list(settings.features.categorical)
            + list(settings.features.forbidden)
        ),
    }
)


`cancellation_reason` and `closed_account_at` occur after the prediction event;
using them would make offline results impressive and the deployed model useless.
Identifiers and timestamps are excluded because they are lineage/context fields,
not learned signals in this contract.

### Exercise

Suppose the retention team can review only 100 accounts per week. Which metric
or threshold constraint would you add, and why?

**Hint:** consider precision among the top `k` scores or a predicted-positive
capacity constraint. Accuracy does not encode capacity.

**Checkpoint:** you can state the positive event, action, horizon, primary
ranking metric, threshold costs, and forbidden information without opening the
test set.

Next: **02_data_quality_and_eda.ipynb**.
